## Resolución de la ecuación de Laplace:
$$\begin{array}{rl}-\Delta u = 1 & \text{en $\Omega$,}\\
u = 0 & \text{en $\partial \Omega$.}\end{array}$$
con $\Omega = [0,1]^2$

### Importamos módulos

In [ ]:
import mfem.ser as mfem
from glvis import glvis

### Mallado

In [ ]:
mesh = mfem.Mesh.MakeCartesian2D(20,20,mfem.Geometry.SQUARE)

### Espacio de elementos finitos

In [ ]:
fec = mfem.H1_FECollection(1,  mesh.Dimension())
fespace = mfem.FiniteElementSpace(mesh, fec)
print('Número de incógnitas:',fespace.GetTrueVSize())

### Formulación variacional
$$\text{Hallar }u\in H^1_0(\Omega):\quad \int_\Omega \nabla u \cdot \nabla v\,dx = \int_\Omega 1\cdot v\,dx,\quad \forall v \in H^1_0(\Omega)$$

In [ ]:
# forma bilineal
a = mfem.BilinearForm(fespace)
a.AddDomainIntegrator(mfem.DiffusionIntegrator())
a.Assemble()

# Segundo miembro (f=1)
one = mfem.ConstantCoefficient(1.0)

b = mfem.LinearForm(fespace)
b.AddDomainIntegrator(mfem.DomainLFIntegrator(one))
b.Assemble()

### Condiciones frontera

#### Etiquetas de la frontera

In [ ]:
boundary_dofs = mfem.intArray()
# Extracción de los grados de libertad asociados a la frontera (al completo)
fespace.GetBoundaryTrueDofs(boundary_dofs)

#### Definimos función para asignar el valor en la frontera

In [ ]:
x = mfem.GridFunction(fespace)
x.Assign(0.0)

### Formulación del sistema

In [ ]:
A = mfem.SparseMatrix()
B = mfem.Vector()
X = mfem.Vector()

a.FormLinearSystem(boundary_dofs, x, b, A, X, B)

###  Resolución del sistema

In [ ]:
mfem.CG(A, B, X, 0, 200, 1e-12, 0.0)

# Asignamos solución a la función grid
a.RecoverFEMSolution(X, b, x)

### Visualización PYGLVIS

In [ ]:
glvis((mesh, x)) 